In [3]:
import numpy as np

def quantize_weights(tensor):
    """Scales a float array to an 8-bit integer and returns the scale factor."""
    max_val = np.max(np.abs(tensor))
    scale_factor = 127.0 / max_val 
    quantized_tensor = np.round(tensor * scale_factor).astype(np.int8)
    return quantized_tensor, scale_factor

def save_to_hex_8bit(array, filename):
    with open(filename, 'w') as f:
        for val in array.flatten():
            f.write(f"{format(int(val) & 0xFF, '02X')}\n")
    print(f"Saved {len(array.flatten())} 8-bit values to {filename}")

def save_to_hex_32bit(array, filename):
    with open(filename, 'w') as f:
        for val in array.flatten():
            # Convert signed int32 to 8-digit 2's complement hex
            f.write(f"{format(int(val) & 0xFFFFFFFF, '08X')}\n")
    print(f"Saved {len(array.flatten())} 32-bit values to {filename}")


hidden_weights, hidden_biases = model.get_layer("hidden_layer").get_weights()
output_weights, output_biases = model.get_layer("output_layer").get_weights()

S_x = 255.0 # Input pixel scale


# PHASE 1

hw_hidden_w, S_w1 = quantize_weights(hidden_weights)

# Phase 1 combined MAC scale
S_mac1 = S_x * S_w1

# Scale Layer 1 bias to match Phase 1 MAC scale
hw_hidden_b_32 = np.round(hidden_biases * S_mac1).astype(np.int32)


# PHASE 2 (No compression/shifting)

# Because Verilog passes the 64-bit RAM directly, Phase 2's input scale IS S_mac1.
hw_output_w, S_w2 = quantize_weights(output_weights)

# Phase 2 combined MAC scale chains Phase 1's massive scale with Phase 2 weights
S_mac2 = S_mac1 * S_w2

# Scale Layer 2 bias to match the massive Phase 2 MAC scale
hw_output_b_32 = np.round(output_biases * S_mac2).astype(np.int32)


# EXPORT

save_to_hex_8bit(hw_hidden_w.T, "hidden_weights.hex") 
save_to_hex_32bit(hw_hidden_b_32, "hidden_biases.hex")  

save_to_hex_8bit(hw_output_w.T, "output_weights.hex") 
save_to_hex_32bit(hw_output_b_32, "output_biases.hex")

Saved 50176 8-bit values to hidden_weights.hex
Saved 64 32-bit values to hidden_biases.hex
Saved 640 8-bit values to output_weights.hex
Saved 10 32-bit values to output_biases.hex


In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32")/255
x_test  = x_test.astype("float32")/255

model = keras.Sequential([
    keras.layers.Input(shape=(28, 28), name="input_layer"),
    keras.layers.Flatten(name="flatten_layer"),
    keras.layers.Dense(64, activation='relu', name="hidden_layer"),
    keras.layers.Dense(10, activation=None, name="output_layer")
])

model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print("Training the software model...")
model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Software Model Accuracy: {test_acc * 100:.2f}%")


Training the software model...
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9159 - loss: 0.2961 - val_accuracy: 0.9516 - val_loss: 0.1610
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9589 - loss: 0.1397 - val_accuracy: 0.9632 - val_loss: 0.1197
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9694 - loss: 0.1022 - val_accuracy: 0.9699 - val_loss: 0.1015
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9754 - loss: 0.0812 - val_accuracy: 0.9689 - val_loss: 0.0972
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9804 - loss: 0.0660 - val_accuracy: 0.9718 - val_loss: 0.0876
313/313 - 1s - 2ms/step - accuracy: 0.9718 - loss: 0.0876
Software Model Accuracy: 97.18%
